# 02 — Operators, Branching & Trigger Rules

Trigger rules are where L5 interviews catch people. Read the table carefully.

## 1. PythonOperator

In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

def process_data(ds, ds_nodash, **kwargs):
    # ds         = "2024-01-15"   (execution date as string)
    # ds_nodash  = "20240115"
    # kwargs["execution_date"] = pendulum datetime object
    print(f"Processing for {ds} | nodash: {ds_nodash}")
    return "done"   # return value auto-pushed to XCom as 'return_value'

with DAG("operator_demo", start_date=datetime(2024,1,1), catchup=False) as dag:
    task = PythonOperator(
        task_id="process",
        python_callable=process_data,
        op_kwargs={"extra": "value"},   # extra kwargs passed to the function
    )

print("PythonOperator created:", task.task_id)

## 2. BranchPythonOperator

In [ ]:
from airflow.operators.python import BranchPythonOperator
from airflow.operators.empty import EmptyOperator

ROW_THRESHOLD = 50_000

def choose_branch(**kwargs):
    # Pull row count from XCom (set by upstream task)
    row_count = kwargs["ti"].xcom_pull(task_ids="count_rows") or 0
    return "full_load" if row_count > ROW_THRESHOLD else "sample_load"

with DAG("branch_demo", start_date=datetime(2024,1,1), catchup=False) as dag:

    count  = PythonOperator(task_id="count_rows",    python_callable=lambda: 75_000)
    branch = BranchPythonOperator(task_id="branch",  python_callable=choose_branch)
    full   = EmptyOperator(task_id="full_load")
    sample = EmptyOperator(task_id="sample_load")

    # CRITICAL: join node needs none_failed_min_one_success
    # Otherwise it waits for BOTH branches and fails because one is always SKIPPED
    join   = EmptyOperator(
        task_id="complete",
        trigger_rule="none_failed_min_one_success",
    )

    count >> branch >> [full, sample] >> join

print("Branch DAG built")

## 3. Trigger Rules — the interview trap

| Rule | Task runs when... | Use case |
|---|---|---|
| `all_success` *(default)* | All upstream succeeded | Normal linear pipeline |
| `all_failed` | All upstream failed | Rollback / cleanup on total failure |
| `all_done` | All upstream finished (any state) | Reporting regardless of outcome |
| `one_failed` | At least one upstream failed | Alert as soon as any task fails |
| `one_success` | At least one upstream succeeded | Take first available result |
| `none_failed` | No upstream failed (success or skipped OK) | After optional steps |
| `none_failed_min_one_success` | No failures AND at least one success | **Branch join — use this** |

**The branch join trap:**
```
branch → full_load  ─┐
       → sample_load ─┤→ complete
```
With default `all_success`: complete waits for both branches. One is always SKIPPED → complete FAILS.
Fix: `trigger_rule="none_failed_min_one_success"`

In [ ]:
# Cleanup task that runs whether upstream succeeded OR failed
from airflow.operators.empty import EmptyOperator

with DAG("cleanup_demo", start_date=datetime(2024,1,1), catchup=False) as dag:
    process  = EmptyOperator(task_id="process")
    cleanup  = EmptyOperator(task_id="cleanup",  trigger_rule="all_done")    # runs always
    notify   = EmptyOperator(task_id="notify",   trigger_rule="one_failed")  # only on failure

    process >> [cleanup, notify]

print("Trigger rule demo built")

## 4. AWS Operators reference (MWAA)

These are commented out — they need real AWS credentials. Know the API.

In [ ]:
# ── GlueJobOperator ──────────────────────────────────────────────
# from airflow.providers.amazon.aws.operators.glue import GlueJobOperator
# GlueJobOperator(
#     task_id="run_glue",
#     job_name="asin-classification",
#     script_args={"--date": "{{ ds }}", "--env": "prod"},
#     aws_conn_id="aws_default",
#     wait_for_completion=True,   # blocks until job finishes
# )

# ── BatchOperator ────────────────────────────────────────────────
# from airflow.providers.amazon.aws.operators.batch import BatchOperator
# BatchOperator(
#     task_id="run_batch",
#     job_name="feature-extraction",
#     job_definition="my-job-def:3",
#     job_queue="ml-queue",
#     overrides={
#         "command": ["python", "main.py", "--date", "{{ ds }}"],
#         "environment": [{"name": "ENV", "value": "prod"}],
#     },
#     aws_conn_id="aws_default",
# )

# ── S3KeySensor ──────────────────────────────────────────────────
# from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
# S3KeySensor(
#     task_id="wait_for_input",
#     bucket_name="my-bucket",
#     bucket_key="input/{{ ds_nodash }}/data.parquet",
#     mode="reschedule",    # releases worker slot between pokes — always use this
#     poke_interval=60,
#     timeout=3600,
#     soft_fail=True,       # SKIPPED on timeout instead of FAILED
#     aws_conn_id="aws_default",
# )

print("AWS operator patterns shown above (commented — needs MWAA env)")